In [ ]:
import pandas as pd
import spatialdata
import sopa
import anndata
import pathlib as pl

import scanpy as sc

import numpy as np

from tqdm.notebook import tqdm

# Get marker genes

In [ ]:
adata_reference = anndata.read_h5ad("/add/path/here/full_cohort.h5ad")

refined_annotations = pd.read_csv("/add/path/here/refined_annotations.csv",index_col=0)

refined_annotations.columns = ["refined_annotations"]

highlevel_refined = {"Hepatocyte": "Epithelial", 
                     "Carcinoma": "Carcinoma", 
                     "Fibroblast": "Fibroblast", 
                     "Quiescent endothelial cells": "Endothelial", 
                     "Smooth muscle": "Muscle", 
                     "Skeletal muscle": "Muscle",
                     "TAM2": "Myeloid", "TAM3": "Myeloid",
                     "TCD4": "Lymphoid", 
                     "Inflammatory CAF": "Fibroblast", 
                     "Adipose CAF": "Fibroblast",
                     "HGF-CAF": "Fibroblast",
                     "TAM1": "Myeloid", 
                     "Myeloid-HighMT": "Unknown/technical", 
                     "Angiogenic EC": "Endothelial", 
                     "Quiescent EC": "Endothelial", 
                     "Venous EC": "Endothelial",
                     "TCD8": "Lymphoid", 
                     "B": "Lymphoid", 
                     "DC": "Myeloid", 
                     "Hepatic EC": "Endothelial", 
                     "Kupffer cells": "Myeloid", 
                     "NK": "Lymphoid", 
                     "Treg": "Lymphoid", 
                     "StrMus-HighMT": "Unknown/technical", 
                     "T-HighMT": "Unknown/technical", 
                     "Mast": "Myeloid", 
                     "Adipocytes": "Stromal/Muscle", 
                     "Endo-HighMT": "Unknown/technical"}

adata_reference.obs = pd.concat([adata_reference.obs,refined_annotations],axis=1)
adata_reference.obs["highlevel_refined"] = adata_reference.obs.refined_annotations.replace(highlevel_refined)

adata_reference = adata_reference[~adata_reference.obs["refined_annotations"].isin(["Hepatocyte","Unknown/technical",
                                                                      "HGF-CAF","Myeloid-HighMT",
                                                                      "T-HighMT","Endothelial",'Kupffer cells',"Hepatic EC"])].copy()

In [ ]:
sc.tl.rank_genes_groups(adata_reference, groupby="refined_annotations", use_raw=False)
markers_df = pd.DataFrame(adata_reference.uns["rank_genes_groups"]["names"]).iloc[0:100, :]
markers = list(np.unique(markers_df.melt().value.values))
print(f"Using {len(markers)} markers")

# P4

In [ ]:
adata = sc.read_h5ad("/add/path/here/Xenium/processed/Xenium_P4_annot.h5ad")

## Annotate

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden")

In [ ]:
group_markers = {}
for gr in adata.obs.leiden.unique():
    group_markers[gr] = sc.get.rank_genes_groups_df(adata, group=gr)

In [ ]:
for gr in sorted(group_markers):
    print(gr)
    print("_________________________________")
    for ct in markers_df:
        
        cmn = np.intersect1d(markers_df[ct].ravel(),group_markers[gr].head(75).names.str.upper().ravel())
        if len(cmn)>0:
            print(ct)
            print(cmn)

In [ ]:
ct_mapping = {"0": "Carcinoma", "1": "DC", "2": "Inflammatory CAF", 
      "3": "Fibroblast", "4": "Carcinoma", "5": "Carcinoma", 
      "6": "T", "7": "TAM2", "8": "Nerve/adrenal", 
      "9": "B",
      "10": "Endothelial", "11": "TAM1",
      "12": "Inflammatory CAF", "13": "Muscle", "14": "Epithelial", }

In [ ]:
adata.obs["Cell_type"] = adata.obs.leiden.replace(ct_mapping)

In [ ]:
sc.pl.umap(adata, color=["Cell_type","tangram_ct_pred"])

In [ ]:
adata.write_h5ad("/add/path/here/Xenium/processed/Xenium_P4_annot.h5ad")

# P10

In [ ]:
adata = sc.read_h5ad("/add/path/here/Xenium/processed/Xenium_P10_annot.h5ad")

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden")

In [ ]:
group_markers = {}
for gr in adata.obs.leiden.unique():
    group_markers[gr] = sc.get.rank_genes_groups_df(adata, group=gr)

In [ ]:
for gr in sorted(group_markers):
    print(gr)
    print("_________________________________")
    for ct in markers_df:
        
        cmn = np.intersect1d(markers_df[ct].ravel(),group_markers[gr].head(75).names.str.upper().ravel())
        if len(cmn)>0:
            print(ct)
            print(cmn)

In [ ]:
ct_mapping = {"0": "Carcinoma", "1": "Carcinoma", "2": "TAM2", 
      "3": "Epithelial", "4": "Carcinoma", "5": "T", 
      "6": "Carcinoma", "7": "T", "8": "Epithelial", 
      "9": "Inflammatory CAF",
      "10": "TAM2", "11": "Inflammatory CAF",
      "12": "Inflammatory CAF", "13": "T", "14": "Endothelial", "15": "TAM1", "16": "DC", "17": "Endothelial", 
              "18": "TAM2", "19": "Treg",
    }

In [ ]:
adata.obs["Cell_type"] = adata.obs.leiden.replace(ct_mapping)

In [ ]:
sc.pl.umap(adata, color=["Cell_type",])

In [ ]:
adata.write_h5ad("/add/path/here/Xenium/processed/Xenium_P10_annot.h5ad")

# P8

In [ ]:
adata = sc.read_h5ad("/add/path/here/Xenium/processed/Xenium_P8_annot.h5ad")

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden")

In [ ]:
group_markers = {}
for gr in adata.obs.leiden.unique():
    group_markers[gr] = sc.get.rank_genes_groups_df(adata, group=gr)

In [ ]:
for gr in sorted(group_markers):
    print(gr)
    print("_________________________________")
    for ct in markers_df:
        
        cmn = np.intersect1d(markers_df[ct].ravel(),group_markers[gr].head(75).names.str.upper().ravel())
        if len(cmn)>0:
            print(ct)
            print(cmn)

In [ ]:
ct_mapping = {"0": "Inflammatory CAF", "1": "Inflammatory CAF", "2": "TAM1", 
      "3": "B", "4": "TAM2", "5": "Muscle", 
      "6": "Muscle", "7": "Treg", "8": "Carcinoma", 
      "9": "Fibroblast",
      "10": "Endothelial", "11": "Muscle",
      "12": "Adipose CAF", "13": "Inflammatory CAF", "14": "DC", "15": "Carcinoma", "16": "Inflammatory CAF", "17": "Muscle", 
              "18": "Adipose CAF", "19": "B", 
              "20": "Adipocyte", "21": "TAM1", "22": "B", "23": "Inflammatory CAF", 
              "24": "Carcinoma", "25": "T", "26": "TAM2", "27": "B", "28": "Nerve/adrenal",
              "29": "Carcinoma", "30": "Mast", "31": "TAM2", "32": "TAM2", "33": "TAM1", 
    }

In [ ]:
adata.obs["Cell_type"] = adata.obs.leiden.replace(ct_mapping)

In [ ]:
sc.pl.umap(adata, color=["Cell_type",])

In [ ]:
adata.write_h5ad("/add/path/here/Xenium/processed/Xenium_P8_annot.h5ad")

# P11

In [ ]:
adata = sc.read_h5ad("/add/path/here/Xenium/processed/Xenium_P11_annot.h5ad")

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden")

In [ ]:
group_markers = {}
for gr in adata.obs.leiden.unique():
    group_markers[gr] = sc.get.rank_genes_groups_df(adata, group=gr)

In [ ]:
for gr in sorted(group_markers):
    print(gr)
    print("_________________________________")
    for ct in markers_df:
        
        cmn = np.intersect1d(markers_df[ct].ravel(),group_markers[gr].head(75).names.str.upper().ravel())
        if len(cmn)>0:
            print(ct)
            print(cmn)

In [ ]:
ct_mapping = {"0": "Epithelial", "1": "Epithelial", "2": "Epithelial (Pre-cancerous)", 
      "3": "Epithelial (Pre-cancerous)", "4": "TAM1", "5": "B", 
      "6": "TAM1", "7": "Carcinoma", "8": "Muscle", 
      "9": "Carcinoma",
      "10": "Carcinoma", "11": "Carcinoma",
      "12": "TAM2", "13": "Inflammatory CAF", "14": "B", "15": "Endothelial", "16": "B", "17": "Nerve/adrenal", 
    }

In [ ]:
adata.obs["Cell_type"] = adata.obs.leiden.replace(ct_mapping)

In [ ]:
sc.pl.umap(adata, color=["Cell_type",])

In [ ]:
adata.write_h5ad("/add/path/here/Xenium/processed/Xenium_P11_annot.h5ad")

# P12

In [ ]:
adata = sc.read_h5ad("/add/path/here/Xenium/processed/Xenium_P12_annot.h5ad")

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden")

In [ ]:
group_markers = {}
for gr in adata.obs.leiden.unique():
    group_markers[gr] = sc.get.rank_genes_groups_df(adata, group=gr)

In [ ]:
for gr in sorted(group_markers):
    print(gr)
    print("_________________________________")
    for ct in markers_df:
        
        cmn = np.intersect1d(markers_df[ct].ravel(),group_markers[gr].head(75).names.str.upper().ravel())
        if len(cmn)>0:
            print(ct)
            print(cmn)

In [ ]:
ct_mapping = {"0": "Epithelial", "1": "Epithelial", "2": "Carcinoma", 
      "3": "Carcinoma", "4": "Carcinoma", "5": "Inflammatory CAF", 
      "6": "Carcinoma", "7": "Carcinoma", "8": "TAM2", 
      "9": "Epithelial",
      "10": "Epithelial", "11": "T",
      "12": "Carcinoma", "13": "Carcinoma", "14": "Endothelial", 
    }

In [ ]:
adata.obs["Cell_type"] = adata.obs.leiden.replace(ct_mapping)

In [ ]:
sc.pl.umap(adata, color=["Cell_type",])

In [ ]:
adata.write_h5ad("/add/path/here/Xenium/processed/Xenium_P12_annot.h5ad")

# P13

In [ ]:
adata = sc.read_h5ad("/add/path/here/Xenium/processed/Xenium_P13_annot.h5ad")

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

sc.tl.leiden(adata, flavor="igraph", n_iterations=2)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden")

In [ ]:
group_markers = {}
for gr in adata.obs.leiden.unique():
    group_markers[gr] = sc.get.rank_genes_groups_df(adata, group=gr)

In [ ]:
for gr in sorted(group_markers):
    print(gr)
    print("_________________________________")
    for ct in markers_df:
        
        cmn = np.intersect1d(markers_df[ct].ravel(),group_markers[gr].head(75).names.str.upper().ravel())
        if len(cmn)>0:
            print(ct)
            print(cmn)

In [ ]:
ct_mapping = {"0": "Carcinoma", "1": "DC", "2": "TAM2", #"2": "Carcinoma", 
      "3": "TAM1", "4": "TAM2", "5": "TAM2", 
      "6": "TAM1", "7": "TAM2", "8": "DC", #"8": "Carcinoma", 
      "9": "DC",
      "10": "TAM2", "11": "Muscle",
      "12": "Endothelial", "13": "DC", #"13": "Carcinoma",
              "14": "TAM2", "15": "TAM2", "16": "TAM2", "17": "TAM2", 
              "18": "Inflammatory CAF", "19": "TAM1", 
              "20": "TAM2", "21": "DC", "22": "Carcinoma", "23": "Carcinoma", 
              "24": "Endothelial", "25": "Epithelial", 
    }

In [ ]:
adata.obs["Cell_type"] = adata.obs.leiden.replace(ct_mapping)

In [ ]:
sc.pl.umap(adata, color=["Cell_type",])

In [ ]:
adata.write_h5ad("/add/path/here/Xenium/processed/Xenium_P13_annot.h5ad")